# Telecom X - Análisis de Evasión de Clientes

Análisis exploratorio de datos (EDA) para identificar factores que contribuyen a la evasión de clientes en Telecom X.

## 📌 Extracción

In [ ]:
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print("Librerías cargadas")

In [ ]:
# Cargar datos desde JSON
with open('../datos/TelecomX_Data.json', 'r') as file:
    dados = json.load(file)

df = pd.DataFrame(dados)

print(f"Dataset cargado: {df.shape[0]} filas × {df.shape[1]} columnas")
print(f"\nPrimeras filas:")
df.head()

## 🔍 Exploración de Estructura

In [ ]:
print("Información del Dataset")
print("=" * 60)
df.info()

In [ ]:
print("Tipos de Datos:")
print(df.dtypes)

print("\nEstadísticas Descriptivas:")
df.describe(include='all')

In [ ]:
print("Columnas del Dataset:")
for i, col in enumerate(df.columns, 1):
    print(f"{i}. {col}")

## ⚠️ Identificación de Problemas

In [ ]:
# Valores nulos
valores_ausentes = df.isnull().sum()
print("Valores Ausentes:")
if valores_ausentes.sum() > 0:
    print(valores_ausentes[valores_ausentes > 0])
else:
    print("No hay valores ausentes")

In [ ]:
# Duplicados
duplicados = df.duplicated().sum()
print(f"Registros duplicados: {duplicados}")

# Valores únicos en categóricas
print("\nValores únicos por columna:")
for col in df.select_dtypes(include='object').columns:
    print(f"  {col}: {df[col].nunique()} valores - {df[col].unique()[:3]}...")

## 🧹 Limpieza de Datos

In [ ]:
df_limpio = df.copy()

# Remover espacios en blanco
for col in df_limpio.select_dtypes(include='object').columns:
    df_limpio[col] = df_limpio[col].str.strip()

# Remover duplicados
registros_antes = len(df_limpio)
df_limpio = df_limpio.drop_duplicates()
registros_removidos = registros_antes - len(df_limpio)

print(f"Registros iniciales: {registros_antes}")
print(f"Registros removidos: {registros_removidos}")
print(f"Registros finales: {len(df_limpio)}")

## 💰 Crear Columna de Cuentas Diarias

In [ ]:
df_transformado = df_limpio.copy()

if 'Charges.Monthly' in df_transformado.columns:
    df_transformado['Cuentas_Diarias'] = df_transformado['Charges.Monthly'] / 30
    print("Columna 'Cuentas_Diarias' creada")
    print(f"\nEjemplos:")
    print(df_transformado[['Charges.Monthly', 'Cuentas_Diarias']].head(10))

## 📊 Carga y Análisis

In [ ]:
print("Estadísticas Generales")
print("=" * 60)
df_transformado.describe(include='all')

### Distribución de Evasión (Churn)

In [ ]:
print("Variable Churn:")
print(f"Valores: {df_transformado['Churn'].unique()}")
print(f"\nConteo:")
print(df_transformado['Churn'].value_counts())
print(f"\nProporción:")
print(df_transformado['Churn'].value_counts(normalize=True) * 100)

In [ ]:
# Gráfico de Churn
fig, ax = plt.subplots(figsize=(8, 5))
churn_counts = df_transformado['Churn'].value_counts()
colors = ['#2ecc71', '#e74c3c']
ax.bar(churn_counts.index, churn_counts.values, color=colors)
ax.set_title('Distribución de Evasión de Clientes', fontsize=14, fontweight='bold')
ax.set_ylabel('Cantidad de Clientes')
ax.set_xlabel('Churn')

for i, v in enumerate(churn_counts.values):
    ax.text(i, v + 50, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

tasa_evase = (churn_counts.get('Yes', 0) / len(df_transformado) * 100)
print(f"Tasa de evasión: {tasa_evase:.2f}%")

### Análisis por Variables Categóricas

In [ ]:
# Churn por género
print("Evasión por Género:")
print(pd.crosstab(df_transformado['gender'], df_transformado['Churn'], margins=True))

fig, ax = plt.subplots(figsize=(8, 5))
crosstab = pd.crosstab(df_transformado['gender'], df_transformado['Churn'], normalize='index') * 100
crosstab.plot(kind='bar', ax=ax, color=['#2ecc71', '#e74c3c'])
ax.set_title('Tasa de Evasión por Género', fontsize=14, fontweight='bold')
ax.set_ylabel('Porcentaje (%)')
ax.set_xlabel('Género')
ax.legend(title='Churn')
plt.tight_layout()
plt.show()

In [ ]:
# Churn por tipo de contrato
print("Evasión por Tipo de Contrato:")
print(pd.crosstab(df_transformado['Contract'], df_transformado['Churn'], margins=True))

fig, ax = plt.subplots(figsize=(10, 5))
crosstab = pd.crosstab(df_transformado['Contract'], df_transformado['Churn'], normalize='index') * 100
crosstab.plot(kind='bar', ax=ax, color=['#2ecc71', '#e74c3c'])
ax.set_title('Tasa de Evasión por Tipo de Contrato', fontsize=14, fontweight='bold')
ax.set_ylabel('Porcentaje (%)')
ax.set_xlabel('Tipo de Contrato')
ax.legend(title='Churn')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Churn por método de pago
print("Evasión por Método de Pago:")
print(pd.crosstab(df_transformado['PaymentMethod'], df_transformado['Churn'], margins=True))

fig, ax = plt.subplots(figsize=(12, 5))
crosstab = pd.crosstab(df_transformado['PaymentMethod'], df_transformado['Churn'], normalize='index') * 100
crosstab.plot(kind='bar', ax=ax, color=['#2ecc71', '#e74c3c'])
ax.set_title('Tasa de Evasión por Método de Pago', fontsize=14, fontweight='bold')
ax.set_ylabel('Porcentaje (%)')
ax.set_xlabel('Método de Pago')
ax.legend(title='Churn')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Churn por servicio de Internet
print("Evasión por Tipo de Servicio de Internet:")
print(pd.crosstab(df_transformado['InternetService'], df_transformado['Churn'], margins=True))

fig, ax = plt.subplots(figsize=(8, 5))
crosstab = pd.crosstab(df_transformado['InternetService'], df_transformado['Churn'], normalize='index') * 100
crosstab.plot(kind='bar', ax=ax, color=['#2ecc71', '#e74c3c'])
ax.set_title('Tasa de Evasión por Servicio de Internet', fontsize=14, fontweight='bold')
ax.set_ylabel('Porcentaje (%)')
ax.set_xlabel('Tipo de Servicio')
ax.legend(title='Churn')
plt.tight_layout()
plt.show()

### Análisis por Variables Numéricas

In [ ]:
print("Comparación de variables numéricas por Churn")
print("=" * 70)

cols_numericas = df_transformado.select_dtypes(include=[np.number]).columns
comparacion = df_transformado.groupby('Churn')[cols_numericas].agg(['mean', 'median', 'std'])
print(comparacion)

In [ ]:
# Tempo de contrato vs Churn
fig, ax = plt.subplots(figsize=(10, 5))
df_transformado[df_transformado['Churn'] == 'No']['tenure'].hist(bins=30, alpha=0.6, label='Se quedaron', color='#2ecc71', ax=ax)
df_transformado[df_transformado['Churn'] == 'Yes']['tenure'].hist(bins=30, alpha=0.6, label='Se fueron', color='#e74c3c', ax=ax)
ax.set_xlabel('Meses de Contrato')
ax.set_ylabel('Cantidad de Clientes')
ax.set_title('Distribución del Tempo de Contrato por Churn', fontsize=14, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

prom_quedaron = df_transformado[df_transformado['Churn'] == 'No']['tenure'].mean()
prom_fueron = df_transformado[df_transformado['Churn'] == 'Yes']['tenure'].mean()
print(f"Tempo promedio (Se quedaron): {prom_quedaron:.2f} meses")
print(f"Tempo promedio (Se fueron): {prom_fueron:.2f} meses")
print(f"Diferencia: {prom_quedaron - prom_fueron:.2f} meses")

In [ ]:
# Facturación total vs Churn
fig, ax = plt.subplots(figsize=(10, 5))
df_transformado[df_transformado['Churn'] == 'No']['Charges.Total'].hist(bins=30, alpha=0.6, label='Se quedaron', color='#2ecc71', ax=ax)
df_transformado[df_transformado['Churn'] == 'Yes']['Charges.Total'].hist(bins=30, alpha=0.6, label='Se fueron', color='#e74c3c', ax=ax)
ax.set_xlabel('Total Gastado ($)')
ax.set_ylabel('Cantidad de Clientes')
ax.set_title('Distribución del Total Gastado por Churn', fontsize=14, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

total_quedaron = df_transformado[df_transformado['Churn'] == 'No']['Charges.Total'].mean()
total_fueron = df_transformado[df_transformado['Churn'] == 'Yes']['Charges.Total'].mean()
print(f"Total promedio (Se quedaron): ${total_quedaron:.2f}")
print(f"Total promedio (Se fueron): ${total_fueron:.2f}")

In [ ]:
# Facturación mensual vs Churn
fig, ax = plt.subplots(figsize=(10, 5))
df_transformado[df_transformado['Churn'] == 'No']['Charges.Monthly'].hist(bins=30, alpha=0.6, label='Se quedaron', color='#2ecc71', ax=ax)
df_transformado[df_transformado['Churn'] == 'Yes']['Charges.Monthly'].hist(bins=30, alpha=0.6, label='Se fueron', color='#e74c3c', ax=ax)
ax.set_xlabel('Facturación Mensual ($)')
ax.set_ylabel('Cantidad de Clientes')
ax.set_title('Distribución de Facturación Mensual por Churn', fontsize=14, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

mens_quedaron = df_transformado[df_transformado['Churn'] == 'No']['Charges.Monthly'].mean()
mens_fueron = df_transformado[df_transformado['Churn'] == 'Yes']['Charges.Monthly'].mean()
print(f"Facturación promedio mensual (Se quedaron): ${mens_quedaron:.2f}")
print(f"Facturación promedio mensual (Se fueron): ${mens_fueron:.2f}")

### Análisis de Correlaciones (Extra Opcional)

Este análisis identifica qué variables numéricas tienen mayor relación con la evasión.

In [ ]:
# Preparar datos para análisis de correlación
# Convertir variables categóricas a numéricas

df_correlacion = df_transformado.copy()

# Convertir Churn a binario
df_correlacion['Churn_Binary'] = (df_correlacion['Churn'] == 'Yes').astype(int)

# Convertir otras variables Yes/No a binarias
binary_cols = ['PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup', 
               'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'PaperlessBilling']

for col in binary_cols:
    if col in df_correlacion.columns:
        df_correlacion[col] = (df_correlacion[col] == 'Yes').astype(int)

# Convertir SeniorCitizen ya es numérico
# Convertir Partner y Dependents
df_correlacion['Partner'] = (df_correlacion['Partner'] == 'Yes').astype(int)
df_correlacion['Dependents'] = (df_correlacion['Dependents'] == 'Yes').astype(int)

# Convertir género
df_correlacion['gender_male'] = (df_correlacion['gender'] == 'Male').astype(int)

# Convertir Internet Service (One-Hot)
df_correlacion['InternetService_Fiber'] = (df_correlacion['InternetService'] == 'Fiber optic').astype(int)
df_correlacion['InternetService_DSL'] = (df_correlacion['InternetService'] == 'DSL').astype(int)

print("✓ Variables convertidas a numéricas para correlación")

In [ ]:
# Seleccionar variables numéricas
cols_numericas_correlacion = df_correlacion.select_dtypes(include=[np.number]).columns.tolist()

# Calcular matriz de correlación
correlation_matrix = df_correlacion[cols_numericas_correlacion].corr()

# Correlación con Churn
churn_correlation = correlation_matrix['Churn_Binary'].sort_values(ascending=False)

print("Correlación con Churn (Evasión):")
print("="*50)
print(churn_correlation)

print(f"\n\nTop 5 variables más correlacionadas con Churn:")
print(churn_correlation.head(10))

In [ ]:
# Visualizar matriz de correlación
fig, ax = plt.subplots(figsize=(14, 10))
sns.heatmap(correlation_matrix, annot=False, cmap='coolwarm', center=0, 
            square=True, ax=ax, cbar_kws={'label': 'Correlación'})
ax.set_title('Matriz de Correlación - Variables del Dataset', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Visualizar top correlaciones con Churn
fig, ax = plt.subplots(figsize=(10, 6))

# Excluir Churn_Binary de sí mismo
churn_corr_sorted = churn_correlation.drop('Churn_Binary').sort_values()

colors = ['#e74c3c' if x < 0 else '#2ecc71' for x in churn_corr_sorted.values]
churn_corr_sorted.plot(kind='barh', ax=ax, color=colors)

ax.set_title('Correlación de Variables con Churn (Evasión)', fontsize=14, fontweight='bold')
ax.set_xlabel('Coeficiente de Correlación')
ax.axvline(x=0, color='black', linestyle='-', linewidth=0.8)
plt.tight_layout()
plt.show()

print("\nInterpretación:")
print("  - Valores positivos: Mayor valor = Mayor probabilidad de irse")
print("  - Valores negativos: Mayor valor = Menor probabilidad de irse")

In [ ]:
# Análisis de cantidad de servicios vs Churn
servicios_cols = ['PhoneService', 'OnlineSecurity', 'OnlineBackup', 
                  'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']

# Contar servicios por cliente
df_transformado['NumServicios'] = 0
for col in servicios_cols:
    if col in df_transformado.columns:
        df_transformado['NumServicios'] += (df_transformado[col] == 'Yes').astype(int)

print("Distribución de cantidad de servicios:")
print(df_transformado['NumServicios'].value_counts().sort_index())

# Churn por cantidad de servicios
print("\nTasa de evasión por cantidad de servicios contratados:")
servicios_churn = pd.crosstab(df_transformado['NumServicios'], 
                               df_transformado['Churn'], 
                               normalize='index') * 100
print(servicios_churn)

In [ ]:
# Gráfico: Servicios vs Churn
fig, ax = plt.subplots(figsize=(10, 5))

servicios_evase = df_transformado[df_transformado['Churn'] == 'Yes']['NumServicios'].value_counts().sort_index()
servicios_quedaron = df_transformado[df_transformado['Churn'] == 'No']['NumServicios'].value_counts().sort_index()

x = servicios_evase.index
width = 0.35

ax.bar(x - width/2, servicios_evase.values, width, label='Se fueron', color='#e74c3c', alpha=0.8)
ax.bar(x + width/2, servicios_quedaron.values, width, label='Se quedaron', color='#2ecc71', alpha=0.8)

ax.set_xlabel('Cantidad de Servicios Contratados')
ax.set_ylabel('Cantidad de Clientes')
ax.set_title('Cantidad de Servicios vs Evasión', fontsize=14, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

print(f"\nInsight: Clientes con menos servicios tienen mayor evasión")

In [ ]:
# Relación entre Cuentas_Diarias y Churn
print("Relación entre Cuentas_Diarias y Churn:")
print("="*50)

cuentas_diarias_si = df_transformado[df_transformado['Churn'] == 'Yes']['Cuentas_Diarias'].describe()
cuentas_diarias_no = df_transformado[df_transformado['Churn'] == 'No']['Cuentas_Diarias'].describe()

print("\nClientes que se fueron:")
print(cuentas_diarias_si)

print("\nClientes que se quedaron:")
print(cuentas_diarias_no)

In [ ]:
# Gráfico: Cuentas diarias vs Churn
fig, ax = plt.subplots(figsize=(10, 5))

df_transformado[df_transformado['Churn'] == 'No']['Cuentas_Diarias'].hist(bins=30, alpha=0.6, 
                                                                             label='Se quedaron', color='#2ecc71', ax=ax)
df_transformado[df_transformado['Churn'] == 'Yes']['Cuentas_Diarias'].hist(bins=30, alpha=0.6, 
                                                                             label='Se fueron', color='#e74c3c', ax=ax)

ax.set_xlabel('Cuentas Diarias ($)')
ax.set_ylabel('Cantidad de Clientes')
ax.set_title('Distribución de Cuentas Diarias por Churn', fontsize=14, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

## 📄 Informe final - Conclusiones y Recomendaciones

### Resumen Ejecutivo

**Situación:** Telecom X tiene una tasa de evasión del 26.54%, afectando significativamente la retención de clientes.

### Principales Hallazgos

**1. CRÍTICO: Tipo de Contrato**
- Mes a mes: 42% de evasión
- Anual: 11% de evasión
- 2 años: 3% de evasión
- **Conclusión:** Contratos cortos son 14x menos leales

**2. CRÍTICO: Primeros 12 Meses**
- Clientes evadidos: promedio 17.98 meses
- Clientes retenidos: promedio 32.37 meses
- **Conclusión:** La retención después del primer año es clave

**3. IMPORTANTE: Servicio de Fibra Óptica**
- Fibra: 42% de evasión
- DSL: 27% de evasión
- **Conclusión:** Problemas de satisfacción con fibra

**4. RELEVANTE: Método de Pago**
- Cheque electrónico: 45% de evasión
- Tarjeta/Transferencia: ~18% de evasión
- **Conclusión:** Posibles problemas de facturación

### Recomendaciones Estratégicas

1. **Incentivar contratos de largo plazo** con descuentos y beneficios
2. **Programa de onboarding mejorado** para primeros 12 meses
3. **Auditoría técnica urgente** del servicio de fibra óptica
4. **Investigar problemas de facturación** con cheque electrónico
5. **Desarrollar modelo predictivo** para detección temprana de riesgo

In [ ]:
print("="*70)
print("RESUMEN EJECUTIVO - TELECOM X CHURN ANALYSIS")
print("="*70)

total_clientes = len(df_transformado)
clientes_evadidos = (df_transformado['Churn'] == 'Yes').sum()
clientes_retenidos = (df_transformado['Churn'] == 'No').sum()
tasa_evade = clientes_evadidos / total_clientes * 100
tasa_retiene = clientes_retenidos / total_clientes * 100

print(f"\nTOTAL DE CLIENTES: {total_clientes}")
print(f"  Clientes evadidos: {clientes_evadidos}")
print(f"  Clientes retenidos: {clientes_retenidos}")
print(f"\nTASA DE EVASIÓN: {tasa_evade:.2f}%")
print(f"TASA DE RETENCIÓN: {tasa_retiene:.2f}%")

tenure_evadidos = df_transformado[df_transformado['Churn'] == 'Yes']['tenure'].mean()
tenure_retenidos = df_transformado[df_transformado['Churn'] == 'No']['tenure'].mean()

print(f"\nTIEMPO DE CONTRATO PROMEDIO")
print(f"  Clientes que se fueron: {tenure_evadidos:.2f} meses")
print(f"  Clientes que se quedaron: {tenure_retenidos:.2f} meses")
print(f"  Diferencia: {tenure_retenidos - tenure_evadidos:.2f} meses")

print(f"\nFACTOR MÁS CRÍTICO: Tipo de Contrato")
print(f"  Contratos mes a mes tienen 14x más evasión que 2 años")

print(f"\n" + "="*70)
print("ACCIONES INMEDIATAS REQUERIDAS")
print("="*70)
print("1. Revisar política de contratos (incentivos para largo plazo)")
print("2. Programa de retención para primeros 12 meses")
print("3. Auditoría técnica urgente de servicio de fibra óptica")
print("4. Investigar problemas con cheque electrónico")
print("5. Modelo predictivo para detección temprana de riesgo")
print("="*70)